#### Import

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
from src.gnn.seed import set_seed
from src.gnn.data import load_and_preprocess_data
from src.gnn.models import GraphSAGE
from src.gnn.training import compute_class_weights, train_with_early_stopping, evaluate, print_test_evaluation
from src.gnn.visualization import plot_learning_curves
from src.utils.multiseed import run_multiseed

#### Load data

In [ ]:
data, device = load_and_preprocess_data('../data/processed/elliptic_pyg_data.pt', undirected=False)

#### Define training function for one seed

In [ ]:
def run_single_gcn_seed(seed):
    set_seed(seed)
    print(f"GCN Train - Seed: {seed}")
    
    model = GraphSAGE(
        in_channels=data.x.shape[1],
        hidden_channels=64,
        out_channels=2,
        aggregator_type='mean',
        dropout=0.5
    ).to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, 
        max_lr=0.01,
        steps_per_epoch=1, 
        epochs=300,
        pct_start=0.1,
    )
    weight = compute_class_weights(data, device)
    criterion = torch.nn.CrossEntropyLoss(weight=weight)
    
    save_path = f'../saved_models/graph_sage_best_seed_{seed}.pt'
    history = train_with_early_stopping(
        model, data, optimizer, criterion,
        scheduler=scheduler,
        save_path=save_path,
        num_epochs=300,
        patience=30,
        monitor_metric='f1_ill',
        clip_grad_norm=1.0
    )
    
    plot_learning_curves(history, model_name=f"GraphSAGE (Seed={seed})")
    
    model.load_state_dict(torch.load(save_path))
    model.eval()
    
    print_test_evaluation(model, data, criterion, label=f"GraphSAGE Test - Seed: {seed}")
    
    test_metrics = evaluate(model, data, criterion, data.test_mask)
    return test_metrics

#### Train with multi-seed

In [ ]:
print("Start training GraphSAGE")
results = run_multiseed(run_single_gcn_seed, "GraphSAGE", seeds=[42, 0, 123])